In [ ]:
import pandas as pd
import os
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../data/processed/df_processed.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# 1. Seleccionar features y target
X = df[['gender', 'SeniorCitizen', 'Partner', 'Dependents',
        'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
        'PaymentMethod', 'MonthlyCharges', 'TotalCharges']]

y = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Identificar columnas categóricas y numéricas
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

print(f"Categóricas ({len(cat_cols)}):", cat_cols)
print(f"Numéricas ({len(num_cols)}):", num_cols)

# 3. Aplicar One-Hot Encoding (drop_first=True para evitar multicolinealidad)
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(int)


# 4. Verificar resultado
print(f"\nShape original: {X.shape}")
print(f"Shape después de encoding: {X_encoded.shape}")

Categóricas (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numéricas (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Shape original: (7032, 19)
Shape después de encoding: (7032, 30)


In [ ]:
X_encoded

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29,29,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56,1889,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53,108,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42,1840,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70,151,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,0,24,84,1990,1,1,1,1,0,1,...,0,1,0,1,1,0,1,0,0,1
7028,0,72,103,7362,0,1,1,1,0,1,...,0,1,0,1,1,0,1,1,0,0
7029,0,11,29,346,0,1,1,0,1,0,...,0,0,0,0,0,0,1,0,1,0
7030,1,4,74,306,1,1,0,1,0,1,...,0,0,0,0,0,0,1,0,0,1


| # | Estrategia | Descripción | Muestras generadas |
|---|---|---|---|
| 0 | **Sin resampling** (Baseline) | Datos originales | 5626 train (desbalanceado) |
| 1 | **SMOTE** | Sintéticos por interpolación de vecinos | ~8278 train (balanceado) |
| 2 | **ADASYN** | SMOTE adaptativo (más en zonas difíciles) | ~8278 train (balanceado) |
| 3 | **RandomOverSampler** | Duplica muestras aleatorias del minoritario | ~8278 train (balanceado) |
| 4 | **RandomUnderSampler** | Elimina muestras del mayoritario | ~3738 train (balanceado) |
| 5 | **SMOTETomek** | SMOTE + limpieza de ruido con Tomek Links | Variable |
| 6 | **SMOTEENN** | SMOTE + limpieza con Edited Nearest Neighbours | Variable |

In [ ]:
# Celda: Crear carpetas y guardar datasets

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

base_path = '../data/resampled'
strategies = {
    'original':      (X_train, y_train),
    'smote':         SMOTE(random_state=42).fit_resample(X_train, y_train),
    'adasyn':        ADASYN(random_state=42).fit_resample(X_train, y_train),
    'random_over':   RandomOverSampler(random_state=42).fit_resample(X_train, y_train),
    'random_under':  RandomUnderSampler(random_state=42).fit_resample(X_train, y_train),
    'smote_tomek':   SMOTETomek(random_state=42).fit_resample(X_train, y_train),
    'smote_enn':     SMOTEENN(random_state=42).fit_resample(X_train, y_train),
}

for name, (X_res, y_res) in strategies.items():
    folder = os.path.join(base_path, name)
    os.makedirs(folder, exist_ok=True)
    
    X_res.to_csv(f'{folder}/X_train.csv', index=False)
    pd.Series(y_res).to_csv(f'{folder}/y_train.csv', index=False, header=['Churn'])
    
    print(f"✓ {name}: {len(y_res)} muestras guardadas")

# Guardar test (una sola vez, común para todos)
test_folder = os.path.join(base_path, 'original')
X_test.to_csv(f'{test_folder}/X_test.csv', index=False)
y_test.to_csv(f'{test_folder}/y_test.csv', index=False, header=['Churn'])
print(f"\n✓ Test set guardado en original/")

/Users/nataliabernalgutierrez/Churn-Prediction/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/nataliabernalgutierrez/Churn-Prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nataliabernalgutierrez/Churn-Prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nataliabernalgutierrez/Churn-Prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/nataliabernalgutierrez/Churn-Prediction/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimat

✓ original: 5625 muestras guardadas
✓ smote: 8260 muestras guardadas
✓ adasyn: 8292 muestras guardadas
✓ random_over: 8260 muestras guardadas
✓ random_under: 2990 muestras guardadas
✓ smote_tomek: 7506 muestras guardadas
✓ smote_enn: 4596 muestras guardadas

✓ Test set guardado en original/
